<div style="border-top: 1px solid rgba(128, 128, 128, 0.45); padding-top: 10px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65;">Walkthrough &middot; about 2 min</span></div>

## <span style="font-family: Newsreader, Georgia, serif; font-size: 34px; font-weight: 400;">2. Environment Setup</span>

<div style="font-size: 19px; line-height: 1.5; max-width: 44em;">The repository's model was deployed on HiPerGator, and training was done on an NVIDIA L4.</div>

### 2.1 Dependency versions

| Layer | Packages | Decision | Why |
| --- | --- | --- | --- |
| Python | Python 3.8.20 | Kept | Supports the kept SciPy 1.7.1 and pandas 1.3.3 pins |
| Science and model | NumPy, SciPy, pandas, timm, albumentations | Kept at the repository's exact pins | Data handling and the model code stay on the versions the authors used |
| Training runtime | PyTorch, torchvision, Lightning, Bolts, TorchMetrics | Updated as one compatible set: PyTorch 1.10 → 2.0.1 for CUDA 11.8, Lightning 1.4.8 → 1.9.5 | Drives the L4 with native mixed precision |

In [ ]:
import importlib.metadata as metadata
from html import escape

import tomli  # Python 3.8 has no tomllib


def read_pins(lines):
    """{package: version} from 'name==version' lines; a bare name is unpinned."""
    pins = {}
    for line in lines:
        requirement = line.split("#", 1)[0].strip()
        if requirement:
            name, separator, version = requirement.partition("==")
            pins[name.lower().replace("_", "-")] = version if separator else "unpinned"
    return pins


def installed(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "missing"


upstream = read_pins((REPO_DIR / "requirements.txt").read_text().splitlines())
selected = read_pins(tomli.loads((PROJECT_DIR / "pyproject.toml").read_text())["project"]["dependencies"])
deps = pd.DataFrame([
    {"package": name, "upstream": upstream.get(name, ""), "selected": version, "installed": installed(name),
     "change": ("added" if name not in upstream else "pinned" if upstream[name] == "unpinned"
                else "kept" if upstream[name] == version else "updated")}
    for name, version in selected.items()
])

# Inline-styled rows rather than a DataFrame table, which JupyterLab right-aligns in outputs.
GROUPS = [("updated", "Updated: the training runtime, moved as one set", PALETTE["accent"]),
          ("kept", "Kept at the repository's pin", PALETTE["neg"]),
          ("pinned", "Pinned where the repository gave no version", PALETTE["rule"]),
          ("added", "Added: transitive and notebook pins", PALETTE["rule"])]
WIDTHS = [30, 20, 24, 24]  # percent of the row: package, repository pin, selected, installed


def row(values, style=""):
    spans = "".join(f'<span style="display: inline-block; width: {width}%; vertical-align: top;">'
                    f"{escape(str(value))}</span>" for value, width in zip(values, WIDTHS))
    return f'<div style="padding: 4px 0;{style}">{spans}</div>'


parts = [row(["Package", "Repository pin", "Selected", "Installed"],
             " font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65;")]
for change, title, colour in GROUPS:
    group = deps[deps["change"] == change]
    if group.empty:
        continue
    parts.append(f'<div style="border-top: 3px solid {colour}; margin-top: 8px; padding: 6px 0 2px 0; '
                 f'font-weight: 600;">{escape(title)} ({len(group)})</div>')
    if change == "added":  # many small pins read better as one wrapped line
        parts.append('<div style="padding: 4px 0; opacity: 0.8;">' + " &middot; ".join(
            f"{escape(name)} {escape(version)}" for name, version in zip(group["package"], group["installed"])
        ) + "</div>")
    else:
        parts += [row(values, f' border-top: 1px solid {PALETTE["rule"]};')
                  for values in group[["package", "upstream", "selected", "installed"]].values.tolist()]
display(HTML('<div style="font-size: 14px; max-width: 780px;">' + "".join(parts) + "</div>"))

mismatched = deps[deps["installed"] != deps["selected"]]
lock_file = PROJECT_DIR / "uv.lock"
caption(f"<b>Table 2.1</b> Read live from <code>concept_transformer/requirements.txt</code>, "
        f"<code>pyproject.toml</code> and this kernel. {len(deps) - len(mismatched)} of {len(deps)} installed "
        f"versions match the selection; <code>uv.lock</code> "
        f"{'pins' if lock_file.is_file() else 'is MISSING and should pin'} the full resolved environment.")
assert mismatched.empty, "Installed differs from pyproject.toml for: " + ", ".join(mismatched["package"])

<details><summary style="cursor: pointer; font-size: 15px;"><b>Backup:</b> rebuild this environment from <code>uv.lock</code></summary>

Run once on a HiPerGator compute node before the call. These are preparation steps, not cells to run live.

```bash
cd "$WORK_DIR"   # export WORK_DIR first: the WORK_DIR path from the configuration cell
export UV_PYTHON_INSTALL_DIR="$PWD/.uv-python" UV_CACHE_DIR="$PWD/.uv-cache"
uv python install 3.8.20
uv venv --python 3.8.20 .venv
uv sync --locked --python .venv/bin/python   # --python: never the cluster's loaded Python
.venv/bin/python -m ipykernel install --sys-prefix \
  --name concept_transformer --display-name "Concept Transformer (HPG Python 3.8)"
.venv/bin/jupyter lab
```

No CUDA toolkit module or Apex install is needed: the CUDA 11.8 PyTorch wheels carry their own runtime, and mixed precision is native to PyTorch. `--locked` makes `uv sync` stop if `pyproject.toml` and `uv.lock` disagree.

</details>

### 2.2 Compatibility issues and where they came from

Six problems stood between the published code and a run on HiPerGator. Only one is a bug in the original code; the others came from the cluster, the documentation or our own runtime upgrade.

| Problem | Origin | Resolution |
| --- | --- | --- |
| An environment command picked the cluster's loaded Conda Python 3.14 instead of 3.8.20 | Cluster | `uv sync --python .venv/bin/python` selects the venv's 3.8.20; the configuration cell asserts Python 3.8 |
| Bolts 0.6.0 does not resolve with Lightning 1.9.5 | Our runtime upgrade | Bolts 0.7.0 with Lightning 1.9.5 and TorchMetrics 0.11.4 |
| The original `Trainer` arguments and `accuracy(preds, y)` call do not run on Lightning 1.9.5 and TorchMetrics 0.11.4; `trainer.test()` needs the datamodule | Our runtime upgrade | `accelerator` and `devices`, native 16-bit mixed precision instead of Apex, `task="multiclass"`, an explicit datamodule |
| CUDA is not visible in the notebook | Slurm: a CPU session gets no GPU | CUDA only when one is visible; training requests an L4 through `sbatch` |
| A batch of one image crashes the spatial explanation loss | **Original code**, `ctc/ctc.py` | Loss kept as published; batch 2 for the debug run, 32 for training |
| The README lists NumPy 1.20.3, `requirements.txt` 1.22.0 | Original documentation | Follow `requirements.txt` |

<details><summary style="cursor: pointer; font-size: 15px;"><b>Backup:</b> the runtime patch, before and after</summary>

Abridged from `patches/hpg-runtime.patch` and `patches/debug-test-no-checkpoint.patch`, shown against the upstream code. It changes only how training is launched and measured; the model, losses, optimizer and augmentation are the repository's.

```diff
 # ctc/ctc_model.py: the accuracy metric
-        acc = accuracy(preds, y)
+        acc = accuracy(preds, y, task="multiclass", num_classes=logits.shape[-1])

 # ctc/ctc_model.py: get_trainer(), GPU branch
-                kwargs = {"amp_backend": "apex", "amp_level": "O2", "precision": 16}
-            trainer = pl.Trainer(
-                logger=logger,
-                gpus=-1,
-                auto_select_gpus=True,
-                ...
-                progress_bar_refresh_rate=10,
-                gradient_clip_val=1.0,
-                **kwargs,
-            )
+    use_cuda = torch.cuda.is_available() and not debug
+    return pl.Trainer(
+        accelerator="gpu" if use_cuda else "cpu",
+        devices=1,
+        precision=16 if use_cuda and amp else 32,
+        ...
+        callbacks=list(callbacks) + [ModelSummary(...), TQDMProgressBar(refresh_rate=10)],
+        gradient_clip_val=1.0 if use_cuda and not debug else 0.0,
+    )

 # cvit_cub.py: the final evaluation
-test_results = trainer.test()
+test_results = trainer.test(
+    model=model,
+    datamodule=trainer.datamodule,
+    ckpt_path=None if args.debug else "best",
+)
```

</details>

### 2.3 Live check: packages, imports and a gradient

Run this when someone asks whether the environment really works. It checks that `pip check` finds no broken requirements, the repository's modules import, `cvit_cub.py` parses its arguments in a fresh process, and autograd returns the known gradient on every device this session can see.

<div style="border-top: 3px solid #ac543f; padding-top: 8px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; font-weight: 600;">Live demo</span><span style="font-size: 13.5px; opacity: 0.7;"> &middot; set <code>RUN_CLI_HELP = False</code> to skip the fresh-process check</span></div>

In [ ]:
# Live: package consistency, the repository's imports, and one forward and backward pass per visible device.
RUN_CLI_HELP = True  # also parse cvit_cub.py's arguments in a fresh Python process (re-imports everything)

import subprocess
import time
import warnings

started = time.time()
checks = []
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
checks.append(("pip check", (pip_check.stdout or pip_check.stderr).strip()))

with warnings.catch_warnings():
    # Noise, not problems: Bolts tags modules this project never uses as "under review",
    # and a CPU-only node has no NVIDIA driver for torch to query.
    warnings.filterwarnings("ignore", message="The feature .* is currently marked under review")
    warnings.filterwarnings("ignore", message="Can't initialize NVML")
    import albumentations
    import pytorch_lightning
    import timm
    import torch
    import torchmetrics
    import torchvision
    from pl_bolts.optimizers.lr_scheduler import LinearWarmupCosineAnnealingLR
    from ctc import CTCModel, cub_cvit
    from data import CUB2011Parts

    checks.append(("Imports", f"torch {torch.__version__}, Lightning {pytorch_lightning.__version__}, "
                              f"TorchMetrics {torchmetrics.__version__}, timm {timm.__version__}; "
                              "the repository's ctc and data modules"))
    torch.set_num_threads(CPU_THREADS)
    devices = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
    for device in devices:
        x = torch.arange(4, dtype=torch.float32, device=device, requires_grad=True)
        loss = x.square().sum()  # 0 + 1 + 4 + 9 = 14
        loss.backward()
        assert loss.item() == 14.0
        torch.testing.assert_close(x.grad, 2 * x.detach())  # the derivative of x squared is 2x
        where = torch.cuda.get_device_name(0) if device == "cuda" else f"{CPU_THREADS} CPU threads"
        checks.append((f"Forward and backward, {device}", f"passed on {where}"))
    if devices == ["cpu"]:
        checks.append(("CUDA", "no GPU visible, as expected without a Slurm GPU allocation"))

if RUN_CLI_HELP:
    cli = subprocess.run([sys.executable, "cvit_cub.py", "--help"], cwd=REPO_DIR, capture_output=True, text=True)
    checks.append(("cvit_cub.py --help", "arguments parsed, exit code 0" if cli.returncode == 0
                   else cli.stderr.strip()[-400:]))

failed = pip_check.returncode != 0 or (RUN_CLI_HELP and cli.returncode != 0)
fact_table(checks, title=f"Environment check · {time.time() - started:.0f} s")
assert not failed, "A check failed: see the rows above."